In [ ]:
from Modules.visual_analyser import VisualAnalyzer
from Modules.data_preprocessing import DataPreprocessing
from Modules.image_analyzer import ImageAnalyzer
import pandas as pd
import os
import random
import numpy as np
import networkx as nx
import igraph as ig
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
# recupération des données
dp = DataPreprocessing()
source_path = ""
df_path = os.path
df = dp.read_data(os.path.join(source_path, "post_rehydrated.pickle"), format_="pickle")
dp.parse_dates()
# filtrage des données (On ne garde que les commentaires et quotes pas les originaux)
df = df[df["join_post_post_type"] != "original"].copy()

## Loading clusters csv file

In [ ]:
df_clusters = pd.read_csv('data/final_df_hdbscan_clustering.csv')
df_clusters.drop(columns=['Unnamed: 0', 'id'], axis=1, inplace=True)
df_clusters.head()

In [ ]:
len(df_clusters['unique_dup_img_name'])

In [ ]:
# tableau annexe : Nombre d'image par cluster
# Compter le nombre d'images par cluster
df_to_export_annexe_report = (
    df_clusters.groupby('cluster_hdbscan')['unique_dup_img_name']
    .count()  # compte les lignes uniques par cluster
    .reset_index(name="Nombre d'images")  # transforme l'index en colonne
)

# Renommer la colonne cluster pour l'export
df_to_export_annexe_report.rename(columns={'cluster_hdbscan': 'Cluster'}, inplace=True)

# Affichage
df_to_export_annexe_report.to_csv('rapport/data/info-cluster.csv')


## Analyse des comptes reposters : `pf_account_id`

In [ ]:
df_pf_account_id_exploded = pd.read_csv('data/df_pf_caccount_id_exploded.csv')
df_pf_account_id_exploded.drop(columns=['Unnamed: 0'], axis=1, inplace=True)
df_pf_account_id_exploded.columns

In [ ]:
df_clusters.columns

In [ ]:
len(df_pf_account_id_exploded)

In [ ]:
df_pf_account_id_exploded = df_pf_account_id_exploded.\
    merge(
        right=df_clusters,
        right_on='image_id',
        left_on='image_name',
        how='left'
    )

In [ ]:
df_pf_account_id_exploded['cluster_hdbscan'] = (
    pd.to_numeric(df_pf_account_id_exploded['cluster_hdbscan'], errors='coerce')
    .astype('Int64')  # type entier nullable pandas
)

In [ ]:
vertices_sizes = (
    df_pf_account_id_exploded
    .groupby('pf_account_id')['unique_dup_img_name']
    .size()
)
vertices_sizes.to_dict()

## ANALYSE RESEAU DES CLUSTERS

In [ ]:
import networkx as nx
from itertools import combinations
import pandas as pd

def build_inverse_time_network(df, clusters_id):

    df_filtered = df[df['cluster_hdbscan'].isin(clusters_id)].copy()
    df_filtered['post_created_at'] = pd.to_datetime(df_filtered['post_created_at'])

    G = nx.Graph()

    # Taille des noeuds = nb images publiées
    node_sizes = df_filtered.groupby('pf_account_id')['unique_dup_img_name'].size().to_dict()
    for node, size in node_sizes.items():
        G.add_node(node, size=size)

    for img_id, group in df_filtered.groupby('unique_dup_img_name'):

        group = group.sort_values('post_created_at')

        for row1, row2 in combinations(group.itertuples(), 2):

            acc1 = row1.pf_account_id
            acc2 = row2.pf_account_id

            #if acc1 == acc2:
            #   continue


            delta_days = abs((row1.post_created_at - row2.post_created_at).days)

            weight = 1 / (1 + delta_days)  # <--- inverse du temps

            if G.has_edge(acc1, acc2):
                G[acc1][acc2]['weight'] += weight
            else:
                G.add_edge(acc1, acc2, weight=weight)

    return G


In [ ]:
g = build_inverse_time_network(df=df_pf_account_id_exploded, clusters_id=[56, 57])
g

In [ ]:
G_ig = ig.Graph.TupleList(
    g.edges(data=True),
    weights=True,   # on garde les poids
    directed=False
)

layout = G_ig.layout("lgl")

# Arêtes proportionnelles au poids
# NetworkX → igraph : on simplifie les poids
edges_for_igraph = [
    (u, v, d["weight"]) for u, v, d in g.edges(data=True)
]

G_ig = ig.Graph.TupleList(
    edges_for_igraph,
    weights=True,
    directed=False
)

# Maintenant chaque e["weight"] est un float
edge_width = [e["weight"] * 0.2 for e in G_ig.es]

clusters_id = [56, 57]
df_filtered = df_pf_account_id_exploded[
    df_pf_account_id_exploded['cluster_hdbscan'].isin(clusters_id)
]
# nombre d'images publiées par compte
node_sizes_dict = df_filtered.groupby('pf_account_id')['unique_dup_img_name'].size().to_dict()

# Noeuds proportionnels au nombre d'images publiées
vertex_sizes = [node_sizes_dict.get(v["name"], 1) for v in G_ig.vs]
vertex_sizes = [(s + 8) * 0.5 for s in vertex_sizes]  # facteur de mise à l'échelle

# Tracer
ig.plot(
    G_ig,
    layout=layout,
    vertex_size=vertex_sizes,
    edge_width=edge_width,
    vertex_color="skyblue",
    vertex_label=G_ig.vs["name"],
    # vertex_label_size=12,
    vertex_label_dist=1.2,# labels masqués si dense
    bbox=(1000, 1000)
)


In [ ]:
import igraph as ig
import numpy as np

# --- Préparer les arêtes avec poids ---
edges_for_igraph = [
    (u, v, d["weight"]) for u, v, d in g.edges(data=True)
]

G_ig = ig.Graph.TupleList(
    edges_for_igraph,
    weights=True,
    directed=False
)

# --- Choisir le layout ---
layout = G_ig.layout("lgl")  # Fruchterman-Reingold pour un rendu équilibré

# --- Épaisseur des arêtes avec transparence ---
edge_width = [e["weight"] * 0.15 for e in G_ig.es]
edge_color = [(0, 0, 0, 0.3) for _ in G_ig.es]  # noir semi-transparent

# --- Filtrer les clusters d'intérêt ---
clusters_id = [56, 57]
df_filtered = df_pf_account_id_exploded[
    df_pf_account_id_exploded['cluster_hdbscan'].isin(clusters_id)
]

# --- Taille des nœuds proportionnelle au nombre d'images ---
node_sizes_dict = df_filtered.groupby('pf_account_id')['unique_dup_img_name'].size().to_dict()
vertex_sizes = [node_sizes_dict.get(v["name"], 1) for v in G_ig.vs]
vertex_sizes = [np.log1p(s) * 5 for s in vertex_sizes]  # scaling logarithmique

# --- Couleur des nœuds par cluster ---
cluster_colors = {56: "lightcoral", 57: "lightgreen"}
vertex_colors = [
    cluster_colors.get(
        df_filtered.loc[df_filtered['pf_account_id'] == v["name"], 'cluster_hdbscan'].values[0],
        "lightgray"
    ) if v["name"] in df_filtered['pf_account_id'].values else "lightgray"
    for v in G_ig.vs
]

# --- Labels visibles uniquement pour les nœuds les plus grands ---
vertex_labels = [
    v["name"] if size > 6 else ""
    for v, size in zip(G_ig.vs, vertex_sizes)
]

# --- Tracer le graph ---
ig.plot(
    G_ig,
    layout=layout,
    vertex_size=vertex_sizes,
    vertex_color=vertex_colors,
    vertex_label=vertex_labels,
    edge_width=edge_width,
    edge_color=edge_color,
    margin=80,
    bbox=(1000, 1000),
    vertex_label_size=12,
    vertex_label_dist=1.2,
)


Les cercles indiquent que plusieurs comptes publient plusieurs fois la meme images, et l'epaisseur traduit une forte présence temporelle.

In [ ]:
import numpy as np
import networkx as nx
from itertools import combinations
import pandas as pd

def build_temporal_weighted_network(df, clusters_id, lambda_decay=0.3):

    df_filtered = df[df['cluster_hdbscan'].isin(clusters_id)].copy()
    df_filtered['post_created_at'] = pd.to_datetime(df_filtered['post_created_at'])

    G = nx.Graph()

    node_sizes = (
        df_filtered
        .groupby('pf_account_id')['unique_dup_img_name']
        .size()
        .to_dict()
    )

    for node, size in node_sizes.items():
        G.add_node(node, size=size)

    for img_id, group in df_filtered.groupby('unique_dup_img_name'):

        group = group.sort_values('post_created_at')

        for row1, row2 in combinations(group.itertuples(), 2):

            acc1 = row1.pf_account_id
            acc2 = row2.pf_account_id

            delta_days = abs(
                (row1.post_created_at - row2.post_created_at)
                .total_seconds()
            ) / 86400

            weight = np.exp(-lambda_decay * delta_days)

            if G.has_edge(acc1, acc2):
                G[acc1][acc2]['weight'] += weight
            else:
                G.add_edge(acc1, acc2, weight=weight)

    return G


In [ ]:
g = build_temporal_weighted_network(df=df_pf_account_id_exploded, clusters_id=[56, 57])
print("Nombre de noeuds :", g.number_of_nodes())
print("Nombre d'arêtes :", g.number_of_edges())

In [ ]:
import igraph as ig
import numpy as np

# --- Seed pour reproductibilité ---
np.random.seed(42)

# --- Récupérer les poids NetworkX ---
weights = [d["weight"] for _, _, d in g.edges(data=True)]

# --- Seuil au 30e percentile ---
threshold = np.percentile(weights, 30)

# --- Filtrer les arêtes faibles ---
edges_filtered = [
    (u, v, d["weight"])
    for u, v, d in g.edges(data=True)
    if d["weight"] >= threshold
]

# --- Construire le graphe filtré ---
G_ig = ig.Graph.TupleList(
    edges_filtered,
    weights=True,
    directed=False
)

# --- Supprimer les nœuds isolés ---
# G_ig.delete_vertices([v.index for v in G_ig.vs if G_ig.degree(v) == 0])

# --- Layout Kamada-Kawai ---
layout = G_ig.layout("lgl")

# --- Épaisseur des arêtes ---
edge_width = [e["weight"] * 0.5 for e in G_ig.es]
edge_color = [(0, 0, 0, 0.25) for _ in G_ig.es]

# --- Taille des noeuds ---
clusters_id = [56, 57]
df_filtered = df_pf_account_id_exploded[
    df_pf_account_id_exploded['cluster_hdbscan'].isin(clusters_id)
]

node_sizes_dict = df_filtered.groupby(
    'pf_account_id'
)['unique_dup_img_name'].size().to_dict()

vertex_sizes = [
    np.log1p(node_sizes_dict.get(v["name"], 1)) * 7
    for v in G_ig.vs
]

G_ig = G_ig.simplify(combine_edges="sum")

# --- Export en PNG ---
ig.plot(
    G_ig,
    target="reseau_clusters_56_57_filtre.png",
    layout=layout,
    vertex_size=vertex_sizes,
    vertex_color="#6F0D0D",
    vertex_frame_color="white",
    vertex_frame_width=1.2,
    vertex_label=G_ig.vs["name"],
    vertex_label_size=14,
    vertex_label_dist=1.1,
    edge_width=edge_width,
    edge_color=edge_color,
    margin=60,
    bbox=(1200, 900),
    background="white"
)


In [ ]:
#edge_width = [e["weight"]["weight"]*1 for e in G_ig.es]
edge_width = [e["weight"] * 0.5 for e in G_ig.es]
partition = G_ig.community_multilevel(weights=edge_width)
print("Nombre de clusters détectés :", len(partition))
clusters = {}
for i, cluster in enumerate(partition):
    print(f"Cluster {i+1} : {[G_ig.vs[v]['name'] for v in cluster]}")
    clusters[i] = [G_ig.vs[v]["name"] for v in cluster]

In [ ]:
import igraph as ig
import numpy as np
import random

# ----------------------------
# Détection Louvain
# ----------------------------
partition = G_ig.community_multilevel(weights="weight")

print("Nombre de communautés :", len(partition))

# ----------------------------
# Générer une couleur par communauté
# ----------------------------
random.seed(42)

def random_color():
    return "#{:06x}".format(random.randint(0, 0xFFFFFF))

community_colors = [random_color() for _ in range(len(partition))]

# ----------------------------
# Attribuer communauté + couleur aux nœuds
# ----------------------------
G_ig.vs["community"] = None
G_ig.vs["color"] = None

for i, cluster in enumerate(partition):
    for vertex_id in cluster:
        G_ig.vs[vertex_id]["community"] = i + 1
        G_ig.vs[vertex_id]["color"] = community_colors[i]

# ----------------------------
# Taille fixe des nœuds
# ----------------------------
DEFAULT_NODE_SIZE = 7
G_ig.vs["size"] = DEFAULT_NODE_SIZE

# ----------------------------
# Ajouter le nombre d’images
# ----------------------------
# dictionnaire déjà calculé : node_sizes_dict

G_ig.vs["nb_images"] = [
    node_sizes_dict.get(v["name"], 0)
    for v in G_ig.vs
]

# ----------------------------
# Export GraphML
# ----------------------------
G_ig.write_graphml("graph_louvain_c57-58.graphml")

print("Export terminé : graph_louvain_c57-58.graphml")

In [ ]:
import pandas as pd

# Créer un dataframe avec une ligne par cluster
data = []

for i, cluster in enumerate(partition):
    names = [G_ig.vs[v]['name'] for v in cluster]
    data.append({
        "Cluster": i+1,
        "Comptes": names,          # liste des noms
        "Nombre de comptes": len(names)      # nombre de noeuds
    })

df_clusters_lov_57_58 = pd.DataFrame(data)
# Affichage
df_clusters_lov_57_58.to_csv('rapport/data/info-cluster-57-58-lovain.csv')


In [ ]:
modularity_value = G_ig.modularity(
    partition.membership,
    weights=edge_width
)

print("Modularité :", modularity_value)

In [ ]:
# Copier le graphe
import random
random.seed(42)
G_random = G_ig.copy()

# Rewire en conservant les degrés
G_random.rewire(n=10 * G_random.ecount()) # pour augmenter la proba que le graphe soit suffisamment mélangé

# Détection des communautés
partition_random = G_random.community_multilevel()

# Calcul modularité
mod_random = G_random.modularity(partition_random.membership)

print("Modularité réseau réel :", modularity_value)
print("Modularité réseau aléatoire :", mod_random)


In [ ]:
print("Simple ?", G_ig.is_simple())
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# dataframe avec nodes, clusters et centralités
df_metrics = pd.DataFrame({
    "node": G_ig.vs["name"],
    "cluster": partition.membership,
    "degree": G_ig.degree(),
    "eigenvector": G_ig.eigenvector_centrality(weights="weight"),
    "betweenness": G_ig.betweenness(weights="weight", normalized=True)
})

df_metrics.to_csv('rapport/data/df_metrics_accounts-56-57.csv')

In [ ]:
df_metrics

In [ ]:
data = []

for cluster_id, cluster in enumerate(partition):
    sub = G_ig.subgraph(cluster)

    size = sub.vcount()
    density = sub.density()
    clustering = sub.transitivity_avglocal_undirected()
    assortativy = sub.assortativity_degree()
    if sub.ecount() > 0:
        mean_weight = np.mean(sub.es['weight'])
    else:
        mean_weight = 0

    data.append({
        "cluster": cluster_id+1,
        "size": size,
        "density": density,
        "clustering": clustering,
        "assortativy": assortativy,
        "mean_weight": mean_weight
    })
df_clusters = pd.DataFrame(data)
df_clusters.to_csv('rapport/data/info-metrique-communautes-56-57.csv')
df_clusters

# Mesures globales du réseau

## 1. Densité du réseau

La **densité** mesure la proportion d’arêtes présentes dans le réseau par rapport au nombre maximal d’arêtes possibles.

Pour un graphe non orienté :

$$
D = \frac{2m}{n(n-1)}
$$

où :

- $n$ est le nombre de nœuds  
- $m$ est le nombre d’arêtes  

### Interprétation

- $D = 1$ : graphe complet (tous les nœuds sont connectés entre eux)  
- $D \approx 0$ : réseau très peu connecté (sparse)  

Une faible densité combinée à une forte modularité suggère une structure communautaire marquée.

---

## 2. Coefficient de clustering

Le **coefficient de clustering** mesure la tendance des voisins d’un noeud à être eux-mêmes connectés entre eux.

### Clustering local

Pour un nœud $i$ :

$$
C_i = \frac{2T_i}{k_i (k_i - 1)}
$$

où :

- $T_i$ est le nombre de triangles impliquant le nœud $i$  
- $k_i$ est le degré du nœud  

### Clustering global

Le clustering global correspond à la moyenne des coefficients locaux :

$$
C = \frac{1}{n} \sum_{i=1}^{n} C_i
$$

### Interprétation

- $C$ élevé → présence de nombreuses structures triangulaires, forte cohésion locale  
- $C$ faible → structure plus arborescente ou hiérarchique  

Dans les réseaux sociaux, le clustering est généralement élevé, reflétant des communautés fortement interconnectées.

---

## 3. Assortativité

L’**assortativité** mesure la tendance des nœuds à se connecter à des nœuds similaires.

### Assortativité par degré

Elle mesure si les nœuds de degré élevé tendent à se connecter à d'autres nœuds de degré élevé.

Le coefficient d’assortativité $r$ est compris entre $-1$ et $1$ :

$$
-1 \leq r \leq 1
$$

### Interprétation

- $r > 0$ : réseau assortatif (les nœuds similaires se connectent entre eux)  
- $r = 0$ : absence de corrélation (structure aléatoire)  
- $r < 0$ : réseau disassortatif (les nœuds dissemblables se connectent)  

Les réseaux sociaux sont généralement assortatifs, tandis que les réseaux biologiques et technologiques sont souvent disassortatifs.

---

## Synthèse interprétative

L’analyse combinée de la densité ($D$), du clustering ($C$) et de l’assortativité ($r$) permet de caractériser la structure globale du réseau :

- Une densité faible mais un clustering élevé suggère une organisation en communautés cohésives.
- Une assortativité positive indique que les nœuds fortement connectés tendent à se regrouper.
- Une assortativité négative révèle une structure hiérarchique ou en étoile.

Ces indicateurs permettent d’évaluer si la structure observée diffère significativement d’un réseau aléatoire de même degré.

In [ ]:
density = G_ig.density()
print(density)

In [ ]:
clustering = G_ig.transitivity_avglocal_undirected() # clustering coefficient 0.84 : réseau communautaire, groupes cohésifs
print(clustering)

In [ ]:
assort = G_ig.assortativity_degree() # les hubs se connectent à des hub ?
print(assort)

### CLUSTERS 0

In [ ]:
g = build_temporal_weighted_network(df=df_pf_account_id_exploded, clusters_id=[0])
print("Nombre de noeuds :", g.number_of_nodes())
print("Nombre d'arêtes :", g.number_of_edges())

In [ ]:
import igraph as ig
import numpy as np

# --- Seed pour reproductibilité ---
np.random.seed(42)

# --- Récupérer les poids NetworkX ---
weights = [d["weight"] for _, _, d in g.edges(data=True)]

# --- Seuil au 30e percentile ---
threshold = np.percentile(weights, 30)

# --- Filtrer les arêtes faibles ---
edges_filtered = [
    (u, v, d["weight"])
    for u, v, d in g.edges(data=True)
    if d["weight"] >= threshold
]

# --- Construire le graphe filtré ---
G_ig = ig.Graph.TupleList(
    edges_filtered,
    weights=True,
    directed=False
)

# --- Supprimer les nœuds isolés ---
# G_ig.delete_vertices([v.index for v in G_ig.vs if G_ig.degree(v) == 0])

# --- Layout Kamada-Kawai ---
layout = G_ig.layout("lgl")

# --- Épaisseur des arêtes ---
edge_width = [e["weight"] * 1 for e in G_ig.es]
edge_color = [(0, 0, 0, 0.25) for _ in G_ig.es]

# --- Taille des noeuds ---
clusters_id = [1]
df_filtered = df_pf_account_id_exploded[
    df_pf_account_id_exploded['cluster_hdbscan'].isin(clusters_id)
]

node_sizes_dict = df_filtered.groupby(
    'pf_account_id'
)['unique_dup_img_name'].size().to_dict()

vertex_sizes = [
    np.log1p(node_sizes_dict.get(v["name"], 1)) * 7 + 10
    for v in G_ig.vs
]
G_ig.simplify(combine_edges="sum")

In [ ]:
print("Simple ?", G_ig.is_simple())
print("Densité globale :", G_ig.density())

In [ ]:
#edge_width = [e["weight"]["weight"]*1 for e in G_ig.es]
edge_width = [e["weight"] for e in G_ig.es]
partition = G_ig.community_multilevel(weights=edge_width)
print("Nombre de clusters détectés :", len(partition))
clusters = {}
for i, cluster in enumerate(partition):
    print(f"Cluster {i+1} : {[G_ig.vs[v]['name'] for v in cluster]}")
    clusters[i] = [G_ig.vs[v]["name"] for v in cluster]

In [ ]:
density = G_ig.density()
print(density)

In [ ]:
clustering = G_ig.transitivity_avglocal_undirected() # clustering coefficient 0.84 : réseau communautaire, groupes cohésifs
print(clustering)

In [ ]:
assort = G_ig.assortativity_degree() # les hubs se connectent à des hub ?
print(assort)

In [ ]:
import igraph as ig
import numpy as np
import random

# ----------------------------
# 1️⃣ Détection Louvain
# ----------------------------
partition = G_ig.community_multilevel(weights="weight")

print("Nombre de communautés :", len(partition))

# ----------------------------
# 2️⃣ Générer une couleur par communauté
# ----------------------------
random.seed(42)

def random_color():
    return "#{:06x}".format(random.randint(0, 0xFFFFFF))

community_colors = [random_color() for _ in range(len(partition))]

# ----------------------------
# 3️⃣ Attribuer communauté + couleur aux nœuds
# ----------------------------
G_ig.vs["community"] = None
G_ig.vs["color"] = None

for i, cluster in enumerate(partition):
    for vertex_id in cluster:
        G_ig.vs[vertex_id]["community"] = i + 1
        G_ig.vs[vertex_id]["color"] = community_colors[i]

# ----------------------------
# 4️⃣ Taille fixe des nœuds
# ----------------------------
DEFAULT_NODE_SIZE = 10
G_ig.vs["size"] = DEFAULT_NODE_SIZE

# ----------------------------
# 5️⃣ Ajouter le nombre d’images
# ----------------------------
# dictionnaire déjà calculé : node_sizes_dict

G_ig.vs["nb_images"] = [
    node_sizes_dict.get(v["name"], 0)
    for v in G_ig.vs
]

# ----------------------------
# Export GraphML
# ----------------------------
G_ig.write_graphml("graph_louvain_c0.graphml")

print("Export terminé : graph_louvain_c0.graphml")

In [ ]:
random.seed(42)
modularity_value = G_ig.modularity(
    partition.membership,
    weights=edge_width
)

print("Modularité :", modularity_value)
# Copier le graphe
G_random = G_ig.copy()

# Rewire en conservant les degrés
G_random.rewire(n=10 * G_random.ecount()) # pour augmenter la proba que le graphe soit suffisamment mélangé

# Détection des communautés
partition_random = G_random.community_multilevel()

# Calcul modularité
mod_random = G_random.modularity(partition_random.membership)

print("Modularité réseau réel :", modularity_value)
print("Modularité réseau aléatoire :", mod_random)

In [ ]:
partition.sizes()

In [ ]:
data = []

for cluster_id, cluster in enumerate(partition):
    sub = G_ig.subgraph(cluster)

    size = sub.vcount()
    density = sub.density()
    clustering = sub.transitivity_avglocal_undirected()
    assortativy = sub.assortativity_degree()
    if sub.ecount() > 0:
        mean_weight = np.mean(sub.es['weight'])
    else:
        mean_weight = 0

    data.append({
        "cluster": cluster_id+1,
        "size": size,
        "density": density,
        "clustering": clustering,
        "assortativy": assortativy,
        "mean_weight": mean_weight
    })
df_clusters = pd.DataFrame(data)
df_clusters.to_csv('rapport/data/info-metrique-communautes-1.csv')
df_clusters

In [ ]:
print("Simple ?", G_ig.is_simple())

In [ ]:
G_ig.betweenness(weights="weight", normalized=True)

In [ ]:
G_ig.eigenvector_centrality(weights="weight")

In [ ]:
import pandas as pd

# Calcul betweenness
values = G_ig.betweenness(weights="weight", normalized=True)

# Créer DataFrame
df_bet = pd.DataFrame({
    "Identifiant du compte": G_ig.vs["name"],
    "Intermédiarité": values
})

# Trier et prendre top 10
df_top10 = df_bet.sort_values("Intermédiarité", ascending=False).head(15)

# Sauvegarder en CSV
df_top10.to_csv("rapport/data/top15_betweenness.csv", index=False)

df_top10

In [ ]:
import pandas as pd

# Calcul betweenness
values = G_ig.eigenvector_centrality(weights="weight")

# Créer DataFrame
df_bet = pd.DataFrame({
    "Identifiant du compte": G_ig.vs["name"],
    "Centralité (Valeurs propres)": values
})

# Trier et prendre top 10
df_top10 = df_bet.sort_values("Centralité (Valeurs propres)", ascending=False).head(15)

# Sauvegarder en CSV
df_top10.to_csv("rapport/data/top15_eigen_vect_centrality.csv", index=False)

df_top10

\section{Mesures de réseau et interprétation}

\subsection{Densité du réseau}
La densité mesure le rapport entre le nombre d'arêtes existantes et le nombre maximal d'arêtes possible :

$$
\text{Densité} = \frac{2|E|}{|V|(|V|-1)}
$$

- $|E|$ : nombre d'arêtes  
- $|V|$ : nombre de nœuds  
- Interprétation : proche de 1 → réseau très connecté ; proche de 0 → réseau dispersé.

Pour un réseau par communauté $C$ :

$$
\text{Densité}(C) = \frac{2|E_C|}{|V_C|(|V_C|-1)}
$$

---

\subsection{Coefficient de clustering}
Mesure la tendance des nœuds à former des triangles :

$$
C_v = \frac{2T_v}{k_v(k_v-1)}
$$

- $T_v$ : nombre de triangles passant par le nœud $v$  
- $k_v$ : degré de $v$  
- Interprétation : proche de 1 → forte densité locale, le voisinage est fortement interconnecté.

Coefficient moyen par réseau : $C = \frac{1}{|V|} \sum_{v \in V} C_v$

---

\subsection{Assortativité}
Mesure la tendance des nœuds à se connecter avec des nœuds similaires (ici par degré) :

$$
r = \frac{\sum_{i} j_i k_i - \frac{1}{M} \sum_i \frac{1}{2}(j_i + k_i)^2}{\sum_i \frac{1}{2}(j_i^2 + k_i^2) - \frac{1}{M} \sum_i \frac{1}{2}(j_i + k_i)^2}
$$

- $j_i, k_i$ : degré des nœuds aux extrémités de l'arête $i$  
- $M$ : nombre total d'arêtes  
- Interprétation : $r>0$ → tendance homophile ; $r<0$ → tendance hétérophile.

---

\subsection{Centralité d'intermédiarité (Betweenness)}
Mesure combien de fois un nœud est sur le chemin le plus court entre deux autres :

$$
C_B(v) = \sum_{s \neq v \neq t} \frac{\sigma_{st}(v)}{\sigma_{st}}
$$

- $\sigma_{st}$ : nombre de chemins les plus courts entre $s$ et $t$  
- $\sigma_{st}(v)$ : nombre de chemins les plus courts passant par $v$  
- Interprétation : haute valeur → le nœud est un pont clé entre sous-réseaux.

---

\subsection{Centralité de vecteur propre (Eigenvector)}
Mesure l'influence d'un nœud en fonction de l’importance de ses voisins :

$$
x_i = \frac{1}{\lambda} \sum_{j} A_{ij} x_j
$$

- $A_{ij}$ : matrice d’adjacence  
- $x_i$ : centralité du nœud $i$  
- Interprétation : nœud influent connecté à des nœuds influents.

---

\subsection{Centralité de degré}
Mesure simple du nombre de liens d’un nœud :

$$
C_D(v) = k_v
$$

- $k_v$ : degré du nœud $v$  
- Interprétation : nœud très connecté localement.

---

\subsection{Références}
- Wasserman, S., Faust, K. \textit{Social Network Analysis: Methods and Applications}, 1994.  
- Newman, M. \textit{Networks: An Introduction}, 2010.  
- igraph documentation : \url{https://igraph.org/python/}  

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Exemple : dataframe avec nodes, clusters et centralités
df_metrics = pd.DataFrame({
    "node": G_ig.vs["name"],
    "cluster": partition.membership,  # si tu as déjà Louvain
    "degree": G_ig.degree(),
    "eigenvector": G_ig.eigenvector_centrality(weights="weight"),
    "betweenness": G_ig.betweenness(weights="weight", normalized=True)
})

df_metrics.to_csv('rapport/data/df_metrics_accounts.csv')

In [ ]:
import numpy as np
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import defaultdict
from statsmodels.stats.multitest import multipletests


# ==============================
# 1️⃣ Construction réseau pondéré
# ==============================

def build_temporal_weighted_network(df, lambda_decay=0.3):

    df = df.copy()
    df['post_created_at'] = pd.to_datetime(df['post_created_at'])

    edge_weights = defaultdict(float)

    for img_id, group in df.groupby('unique_dup_img_name'):
        group = group.sort_values('post_created_at')
        rows = list(group.itertuples())

        for i in range(len(rows)):
            for j in range(i + 1, len(rows)):

                acc1 = rows[i].pf_account_id
                acc2 = rows[j].pf_account_id

                delta_days = abs(
                    (rows[i].post_created_at - rows[j].post_created_at)
                    .total_seconds()
                ) / 86400

                weight = np.exp(-lambda_decay * delta_days)

                key = tuple(sorted((acc1, acc2)))
                edge_weights[key] += weight

    return edge_weights


# ==============================
# 2️⃣ Permutation des timestamps
# ==============================

def permute_timestamps(df):
    df_perm = df.copy()
    df_perm['post_created_at'] = np.random.permutation(
        df_perm['post_created_at'].values
    )
    return df_perm


# ==============================
# 3️⃣ Test par permutation
# ==============================

def permutation_test(df, B=100, lambda_decay=0.3):

    print("Building observed network...")
    observed = build_temporal_weighted_network(df, lambda_decay)

    permuted_weights = defaultdict(list)

    print("Running permutations...")
    for b in range(B):
        print(f"Permutation {b+1}/{B}")

        df_perm = permute_timestamps(df)
        perm_net = build_temporal_weighted_network(df_perm, lambda_decay)

        for edge, weight in perm_net.items():
            permuted_weights[edge].append(weight)

        # Assurer que toutes les arêtes observées ont B valeurs
        for edge in observed.keys():
            if edge not in perm_net:
                permuted_weights[edge].append(0.0)

    results = []

    for edge, w_obs in observed.items():

        null_dist = permuted_weights[edge]

        mean_null = np.mean(null_dist)
        std_null = np.std(null_dist)

        if std_null > 0:
            z = (w_obs - mean_null) / std_null
        else:
            z = 0.0

        # p-value empirique (one-sided)
        p_value = np.mean(np.array(null_dist) >= w_obs)

        results.append({
            "account_1": edge[0],
            "account_2": edge[1],
            "W_obs": w_obs,
            "E_null": mean_null,
            "std_null": std_null,
            "Z": z,
            "p_value": p_value
        })

    return pd.DataFrame(results)


# ==============================
# 4️⃣ Correction FDR
# ==============================

def apply_fdr(results_df, alpha=0.05):

    rejected, pvals_corrected, _, _ = multipletests(
        results_df["p_value"],
        alpha=alpha,
        method='fdr_bh'
    )

    results_df["p_adj"] = pvals_corrected
    results_df["significant"] = rejected

    return results_df


# ==============================
# 5️⃣ Construction backbone
# ==============================

def build_significant_graph(results_df):

    G = nx.Graph()

    sig_edges = results_df[results_df["significant"]]

    for _, row in sig_edges.iterrows():
        G.add_edge(
            row["account_1"],
            row["account_2"],
            weight=row["W_obs"],
            z=row["Z"],
            p_adj=row["p_adj"]
        )

    return G


# ==============================
# 🚀 PIPELINE COMPLET
# ==============================

def build_statistical_backbone(df, B=100, lambda_decay=0.3, alpha=0.05):

    results = permutation_test(df, B=B, lambda_decay=lambda_decay)

    results = apply_fdr(results, alpha=alpha)

    G_significant = build_significant_graph(results)

    return results, G_significant

In [ ]:
df_filtered_ = df_pf_account_id_exploded[df_pf_account_id_exploded['cluster_hdbscan']==0]
df_filtered_.head()

In [ ]:
# df_filtered_ = df_filtered[df_filtered['cluster_hdbscan']==0]
results_df, G_backbone = build_statistical_backbone(
    df_filtered_,
    B=200,
    lambda_decay=0.5,
    alpha=0.05
)

In [ ]:
results_df['significant'].dtype

In [ ]:
results_df[
    (results_df['account_1'] != results_df['account_2']) &
    (results_df['significant'] == True)
]

In [ ]:
results_df[
    (results_df['account_1'] != results_df['account_2']) &
    (results_df['significant'] == True)
].head(15).to_csv('rapport/data/stat_c0.csv')

In [ ]:
import igraph as ig

# self.sem_graph = graphe
# Conversion en igraph
G_ig = ig.Graph.TupleList(
    G_backbone.edges(data=True),
    weights=True,  # on garde les poids (similarité)
    directed=False,  # graphe non orienté (pas de direction)
)
layout = G_ig.layout("lgl")  # algorithme de mise en page pour les grands graphes
edge_width = [
    e["weight"]["weight"] * 2 for e in G_ig.es
]  # épaisseur proportionnelle au poids (2 cosim)
len(edge_width)
ig.plot(
    G_ig,
    layout=layout,
    # vertex_label=G_ig.vs["name"],   # id_imagess
    vertex_size=5,
    edge_width=edge_width,
)

In [ ]:
print(G_ig.es.attributes())
for e in G_ig.es:
    if e["weight"] is None or not isinstance(e["weight"], (int, float)):
        e["weight"] = 0.0
    else:
        e["weight"] = float(e["weight"])

In [ ]:
print("Simple ?", G_ig.is_simple())
G_ig.simplify(combine_edges={"weight": "sum"})
print("Simple ?", G_ig.is_simple())

In [ ]:
random.seed(42)
modularity_value = G_ig.modularity(
    partition.membership,
    weights=edge_width
)

print("Modularité :", modularity_value)
# Copier le graphe
G_random = G_ig.copy()

# Rewire en conservant les degrés
G_random.rewire(n=10 * G_random.ecount()) # pour augmenter la proba que le graphe soit suffisamment mélangé

# Détection des communautés
partition_random = G_random.community_multilevel()

# Calcul modularité
mod_random = G_random.modularity(partition_random.membership)

print("Modularité réseau réel :", modularity_value)
print("Modularité réseau aléatoire :", mod_random)

In [ ]:
import igraph as ig
import numpy as np
import random

# ----------------------------
# 1️⃣ Détection Louvain
# ---------------------------------------
# 2️⃣ Générer une couleur par communauté
# ----------------------------
random.seed(42)

def random_color():
    return "#{:06x}".format(random.randint(0, 0xFFFFFF))

community_colors = [random_color() for _ in range(len(partition))]

# ----------------------------
# 3️⃣ Attribuer communauté + couleur aux nœuds
# ----------------------------
G_ig.vs["community"] = None
G_ig.vs["color"] = None

for i, cluster in enumerate(partition):
    for vertex_id in cluster:
        G_ig.vs[vertex_id]["community"] = i + 1
        G_ig.vs[vertex_id]["color"] = community_colors[i]

# ----------------------------
# 4️⃣ Taille fixe des nœuds
# ----------------------------
DEFAULT_NODE_SIZE = 10
G_ig.vs["size"] = DEFAULT_NODE_SIZE

# ----------------------------
# 5️⃣ Ajouter le nombre d’images
# ----------------------------
# dictionnaire déjà calculé : node_sizes_dict

G_ig.vs["nb_images"] = [
    node_sizes_dict.get(v["name"], 0)
    for v in G_ig.vs
]

# ----------------------------
# Export GraphML
# ----------------------------
G_ig.write_graphml("graph_louvain_c0_stat.graphml")

print("Export terminé : graph_louvain_c0_stat.graphml")

# Détection de coordination temporelle dans un cluster d’images

---

## 1️⃣ Données

- Ensemble de **comptes** : $U = \{u_1, u_2, \dots, u_n\}$  
- Ensemble d’**images** : $I = \{i_1, i_2, \dots, i_m\}$  
- Chaque image appartient à un **cluster HDBSCAN** : $C = \{c_1, \dots, c_k\}$  
- Chaque post a un **timestamp** : $t_{u,i}$

On se restreint ici à un **cluster spécifique** $c^*$ pour détecter la coordination :

$$
I_{c^*} = \{ i \in I \mid i \text{ appartient au cluster } c^* \}
$$

---

## 2️⃣ Construction du réseau pondéré

Pour chaque paire de comptes $(u,v)$ ayant posté une même image $i \in I_{c^*}$ :

$$
\Delta t_{uv,i} = | t_{u,i} - t_{v,i} |
$$

On définit un **poids exponentiel** pour valoriser la proximité temporelle :

$$
w_{uv,i} = e^{-\lambda \Delta t_{uv,i}}
$$

où $\lambda > 0$ est un paramètre de décroissance.  

Le **poids total de coordination** entre deux comptes dans ce cluster est :

$$
W_{uv}^{c^*} = \sum_{i \in I_{c^*}} w_{uv,i} = \sum_{i \in I_{c^*}} e^{-\lambda \Delta t_{uv,i}}
$$

---

## 3️⃣ Modèle nul par permutation

Hypothèse nulle $H_0$ : la coordination observée est due au hasard.  

On construit un **modèle nul empirique** en permutant les timestamps $B$ fois.  
Pour chaque permutation $b$ :

$$
t_{u,i}^{(b)} \sim \text{Permutation des timestamps}, \quad b = 1, \dots, B
$$

et on calcule :

$$
W_{uv}^{(b)} = \sum_{i \in I_{c^*}} e^{-\lambda | t_{u,i}^{(b)} - t_{v,i}^{(b)} |}
$$

---

## 4️⃣ Z-score et p-value empirique

Pour chaque paire de comptes $(u,v)$ :

- Moyenne et écart-type sous le modèle nul :

$$
\mu_{uv}^{null} = \frac{1}{B} \sum_{b=1}^{B} W_{uv}^{(b)}, \quad 
\sigma_{uv}^{null} = \sqrt{\frac{1}{B} \sum_{b=1}^{B} \left(W_{uv}^{(b)} - \mu_{uv}^{null}\right)^2}
$$

- Z-score :

$$
Z_{uv} = \frac{W_{uv}^{c^*} - \mu_{uv}^{null}}{\sigma_{uv}^{null}}
$$

- p-value empirique (one-sided) :

$$
p_{uv} = \frac{\#\{ W_{uv}^{(b)} \ge W_{uv}^{c^*} \}}{B}
$$

---

## 5️⃣ Extraction du backbone significatif

- On applique une **correction FDR** (Benjamini-Hochberg) pour les tests multiples  
- On garde seulement les liens significatifs ($p_{uv}^{adj} < \alpha$)  

Le **graphe final** :

- Nœuds = comptes $u \in U$  
- Arêtes = liens significatifs $(u,v)$  
- Poids = $W_{uv}^{c^*}$  
- Z-score et p-value ajustée = indicateurs de coordination statistiquement significative

---

## 6️⃣ Résumé du pipeline

1. **Filtrer le cluster d’image** $c^*$  
2. **Construire le réseau pondéré** par proximité temporelle :

$$
W_{uv}^{c^*} = \sum_{i \in I_{c^*}} e^{-\lambda \Delta t_{uv,i}}
$$

3. **Simuler $B$ permutations** des timestamps → modèle nul  
4. **Calculer Z-scores et p-values empiriques**  
5. **Corriger les p-values** via FDR  
6. **Extraire le backbone significatif** → comptes coordonnés

## CLUSTER 56-57

In [ ]:
df_filtered_ = df_pf_account_id_exploded[df_pf_account_id_exploded['cluster_hdbscan'].isin([56, 57])]
df_filtered_.head()

In [ ]:
results_df, G_backbone = build_statistical_backbone(
    df_filtered_,
    B=200,
    lambda_decay=0.5,
    alpha=0.05
)

In [ ]:
results_df[
    (results_df['account_1'] != results_df['account_2']) &
    (results_df['significant'] == True)
].head(15).to_csv('rapport/data/stat_c5657.csv')

In [ ]:
import igraph as ig

# self.sem_graph = graphe
# Conversion en igraph
G_ig = ig.Graph.TupleList(
    G_backbone.edges(data=True),
    weights=True,  # on garde les poids (similarité)
    directed=False,  # graphe non orienté (pas de direction)
)
layout = G_ig.layout("lgl")  # algorithme de mise en page pour les grands graphes
edge_width = [
    e["weight"]["weight"] * 2 for e in G_ig.es
]  # épaisseur proportionnelle au poids (2 cosim)
len(edge_width)
ig.plot(
    G_ig,
    layout=layout,
    # vertex_label=G_ig.vs["name"],   # id_imagess
    vertex_size=5,
    edge_width=edge_width,
)

In [ ]:
print(G_ig.es.attributes())
for e in G_ig.es:
    if e["weight"] is None or not isinstance(e["weight"], (int, float)):
        e["weight"] = 0.0
    else:
        e["weight"] = float(e["weight"])
print("Simple ?", G_ig.is_simple())
G_ig.simplify(combine_edges={"weight": "sum"})
print("Simple ?", G_ig.is_simple())

In [ ]:
edge_width = [e["weight"] for e in G_ig.es]
partition = G_ig.community_multilevel(weights=edge_width)
print("Nombre de clusters détectés :", len(partition))
clusters = {}
for i, cluster in enumerate(partition):
    print(f"Cluster {i+1} : {[G_ig.vs[v]['name'] for v in cluster]}")
    clusters[i] = [G_ig.vs[v]["name"] for v in cluster]

In [ ]:
random.seed(42)
modularity_value = G_ig.modularity(
    partition.membership,
    weights=edge_width
)

print("Modularité :", modularity_value)
# Copier le graphe
G_random = G_ig.copy()

# Rewire en conservant les degrés
G_random.rewire(n=10 * G_random.ecount()) # pour augmenter la proba que le graphe soit suffisamment mélangé

# Détection des communautés
partition_random = G_random.community_multilevel()

# Calcul modularité
mod_random = G_random.modularity(partition_random.membership)

print("Modularité réseau réel :", modularity_value)
print("Modularité réseau aléatoire :", mod_random)

In [ ]:
import igraph as ig
import numpy as np
import random

# ----------------------------
# 1️⃣ Détection Louvain
# ---------------------------------------
# 2️⃣ Générer une couleur par communauté
# ----------------------------
random.seed(42)

def random_color():
    return "#{:06x}".format(random.randint(0, 0xFFFFFF))

community_colors = [random_color() for _ in range(len(partition))]

# ----------------------------
# 3️⃣ Attribuer communauté + couleur aux nœuds
# ----------------------------
G_ig.vs["community"] = None
G_ig.vs["color"] = None

for i, cluster in enumerate(partition):
    for vertex_id in cluster:
        G_ig.vs[vertex_id]["community"] = i + 1
        G_ig.vs[vertex_id]["color"] = community_colors[i]

# ----------------------------
# 4️⃣ Taille fixe des nœuds
# ----------------------------
DEFAULT_NODE_SIZE = 10
G_ig.vs["size"] = DEFAULT_NODE_SIZE

# ----------------------------
# 5️⃣ Ajouter le nombre d’images
# ----------------------------
# dictionnaire déjà calculé : node_sizes_dict

G_ig.vs["nb_images"] = [
    node_sizes_dict.get(v["name"], 0)
    for v in G_ig.vs
]

# ----------------------------
# Export GraphML
# ----------------------------
G_ig.write_graphml("graph_louvain_c5657_stat.graphml")

print("Export terminé : graph_louvain_c5657_stat.graphml")

## INHAUTHENTICITY : 10 Septembre

In [ ]:
img_cluster_2 = df_pf_account_id_exploded['unique_dup_img_name'][df_pf_account_id_exploded['cluster_hdbscan']==47].tolist()
random.seed(42)
random.shuffle(img_cluster_2)
img_cluster_2_to_display = img_cluster_2
print(img_cluster_2_to_display)
indices = [0, 1, 2, 4, 5, 6, 7, 9, 10, 12, 18, 23, 24, 26, 27, 29, 30, 31, 32, 36]

selected_images = [img_cluster_2_to_display[i] for i in indices]
img_analyzer = ImageAnalyzer(df=df)
img_analyzer.display_specific_imgs(
    from_internet=False,
    source_path=source_path,
    images_dir = ['img', 'source_img'],
    image_names=selected_images,
    ncols=5
)

## Coordinated beahavior

In [ ]:
g = build_temporal_weighted_network(df=df_pf_account_id_exploded, clusters_id=[47])
print("Nombre de noeuds :", g.number_of_nodes())
print("Nombre d'arêtes :", g.number_of_edges())

import igraph as ig
import numpy as np

# --- Seed pour reproductibilité ---
np.random.seed(42)

# --- Récupérer les poids NetworkX ---
weights = [d["weight"] for _, _, d in g.edges(data=True)]

# --- Seuil au 30e percentile ---
threshold = np.percentile(weights, 30)

# --- Filtrer les arêtes faibles ---
edges_filtered = [
    (u, v, d["weight"])
    for u, v, d in g.edges(data=True)
    if d["weight"] >= threshold
]

# --- Construire le graphe filtré ---
G_ig = ig.Graph.TupleList(
    edges_filtered,
    weights=True,
    directed=False
)

# --- Supprimer les nœuds isolés ---
# G_ig.delete_vertices([v.index for v in G_ig.vs if G_ig.degree(v) == 0])

# --- Layout Kamada-Kawai ---
layout = G_ig.layout("lgl")

# --- Épaisseur des arêtes ---
edge_width = [e["weight"] * 1 for e in G_ig.es]
edge_color = [(0, 0, 0, 0.25) for _ in G_ig.es]

# --- Taille des noeuds ---
clusters_id = [1]
df_filtered = df_pf_account_id_exploded[
    df_pf_account_id_exploded['cluster_hdbscan'].isin(clusters_id)
]

node_sizes_dict = df_filtered.groupby(
    'pf_account_id'
)['unique_dup_img_name'].size().to_dict()

vertex_sizes = [
    np.log1p(node_sizes_dict.get(v["name"], 1)) * 7 + 10
    for v in G_ig.vs
]
G_ig.simplify(combine_edges="sum")

print("Simple ?", G_ig.is_simple())
print("Densité globale :", G_ig.density())


In [ ]:
G_ig.write_graphml("graph_louvain_c47.graphml")

print("Export terminé : graph_louvain_c47.graphml")

In [ ]:
#edge_width = [e["weight"]["weight"]*1 for e in G_ig.es]
edge_width = [e["weight"] for e in G_ig.es]
partition = G_ig.community_multilevel(weights=edge_width)
print("Nombre de clusters détectés :", len(partition))
clusters = {}
for i, cluster in enumerate(partition):
    print(f"Cluster {i+1} : {[G_ig.vs[v]['name'] for v in cluster]}")
    clusters[i] = [G_ig.vs[v]["name"] for v in cluster]

In [ ]:
import igraph as ig
import numpy as np
import random

# ----------------------------
# 1️⃣ Détection Louvain
# ----------------------------
partition = G_ig.community_multilevel(weights="weight")

print("Nombre de communautés :", len(partition))

# ----------------------------
# 2️⃣ Générer une couleur par communauté
# ----------------------------
random.seed(42)

def random_color():
    return "#{:06x}".format(random.randint(0, 0xFFFFFF))

community_colors = [random_color() for _ in range(len(partition))]

# ----------------------------
# 3️⃣ Attribuer communauté + couleur aux nœuds
# ----------------------------
G_ig.vs["community"] = None
G_ig.vs["color"] = None

for i, cluster in enumerate(partition):
    for vertex_id in cluster:
        G_ig.vs[vertex_id]["community"] = i + 1
        G_ig.vs[vertex_id]["color"] = community_colors[i]

# ----------------------------
# 4️⃣ Taille fixe des nœuds
# ----------------------------
DEFAULT_NODE_SIZE = 10
G_ig.vs["size"] = DEFAULT_NODE_SIZE

# ----------------------------
# 5️⃣ Ajouter le nombre d’images
# ----------------------------
# dictionnaire déjà calculé : node_sizes_dict

G_ig.vs["nb_images"] = [
    node_sizes_dict.get(v["name"], 0)
    for v in G_ig.vs
]

# ----------------------------
# Export GraphML
# ----------------------------
G_ig.write_graphml("graph_louvain_c47.graphml")

print("Export terminé : graph_louvain_c47.graphml")


In [ ]:
random.seed(42)
modularity_value = G_ig.modularity(
    partition.membership,
    weights=edge_width
)

print("Modularité :", modularity_value)
# Copier le graphe
G_random = G_ig.copy()

# Rewire en conservant les degrés
G_random.rewire(n=10 * G_random.ecount()) # pour augmenter la proba que le graphe soit suffisamment mélangé

# Détection des communautés
partition_random = G_random.community_multilevel()

# Calcul modularité
mod_random = G_random.modularity(partition_random.membership)

print("Modularité réseau réel :", modularity_value)
print("Modularité réseau aléatoire :", mod_random)


In [ ]:

data = []

for cluster_id, cluster in enumerate(partition):
    sub = G_ig.subgraph(cluster)

    size = sub.vcount()
    density = sub.density()
    clustering = sub.transitivity_avglocal_undirected()
    assortativy = sub.assortativity_degree()
    if sub.ecount() > 0:
        mean_weight = np.mean(sub.es['weight'])
    else:
        mean_weight = 0

    data.append({
        "cluster": cluster_id+1,
        "size": size,
        "density": density,
        "clustering": clustering,
        "assortativy": assortativy,
        "mean_weight": mean_weight
    })
df_clusters = pd.DataFrame(data)
df_clusters.to_csv('rapport/data/info-metrique-communautes-47.csv')
df_clusters


import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Exemple : dataframe avec nodes, clusters et centralités
df_metrics = pd.DataFrame({
    "node": G_ig.vs["name"],
    "cluster": partition.membership,  # si tu as déjà Louvain
    "degree": G_ig.degree(),
    "eigenvector": G_ig.eigenvector_centrality(weights="weight"),
    "betweenness": G_ig.betweenness(weights="weight", normalized=True)
})

df_metrics.head()

In [ ]:
df_filtered_ = df_pf_account_id_exploded[df_pf_account_id_exploded['cluster_hdbscan']==47]
df_filtered_.head()


In [ ]:

# df_filtered_ = df_filtered[df_filtered['cluster_hdbscan']==47]
results_df, G_backbone = build_statistical_backbone(
    df_filtered_,
    B=100,
    lambda_decay=0.3,
    alpha=0.05
)


results_df[
    (results_df['account_1'] != results_df['account_2']) &
    (results_df['significant'] == True)
].head(10).to_csv('rapport/data/stat_c47.csv')

In [ ]:
results_df.head()